# Normalize

> Provider payload normalization helpers.

In [ ]:
#| default_exp normalize

## Imports

In [ ]:
#| export
import json
from dataclasses import dataclass, field, fields
from collections import Counter
from fastcore.utils import *
from fastcore.meta import delegates
from fastllm.types import *
from fastspec.errors import api_error_from_event

In [ ]:
#| hide
import yaml
from fastspec.spec import *
from fastspec.oapi import *
from fastcore.test import *
from cachy.core import enable_cachy, disable_cachy

In [ ]:
enable_cachy(hdrs=('content-type',))

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
specs_path = Path('../specs/')

ant_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'anthropic.yml').read_text())))
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))

In [ ]:
ant_spec, oai_spec, gem_spec

(SpecParser(base_url='https://api.anthropic.com', ops=47),
 SpecParser(base_url='https://api.openai.com/v1', ops=241),
 SpecParser(base_url='https://generativelanguage.googleapis.com/', ops=79))

In [ ]:
ant_cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
gem_cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})

In [ ]:
ant_cli.groups.keys()

dict_keys(['messages', 'complete', 'models', 'files', 'skills', 'messages_beta_true', 'models_beta_true', 'files_beta_true', 'skills_beta_true'])

In [ ]:
oai_cli.groups.keys()

dict_keys(['assistants', 'audio', 'batches', 'chat', 'completions', 'containers', 'conversations', 'embeddings', 'evals', 'files', 'fine_tuning', 'images', 'models', 'moderations', 'organization', 'projects', 'realtime', 'responses', 'threads', 'uploads', 'vector_stores', 'videos', 'skills', 'chatkit'])

In [ ]:
gem_cli.groups.keys()

dict_keys(['batches', 'models', 'tuned_models', 'dynamic', 'cached_contents', 'media', 'files', 'generated_files', 'file_search_stores', 'corpora'])

## Tool Calls

All four providers represent tool invocations differently in their wire format:

- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Tool calls are flat output items: `{type: "function_call", call_id, name, arguments}` where `arguments` is a JSON **string**. Streamed via `response.function_call_arguments.delta` events.
- **[OpenAI-compat (Chat)](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Tool calls are nested: `{id, type: "function", function: {name, arguments}}` where `arguments` is a JSON **string**. Streamed in chunks via `tool_calls[i].function.arguments` deltas.
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Tool calls are `tool_use` content blocks: `{id, name, input, type: "tool_use"}` where `input` is a parsed JSON object. Streamed via `input_json_delta` events.
- **[Gemini](https://ai.google.dev/api/generate-content)** — Tool calls are `functionCall` parts: `{id, name, args}` where `args` is a parsed JSON object. Not chunked during streaming.

> **Spec sources:** OpenAI Responses `FunctionToolCall` (`specs/openai.with-code-samples.yml:44347–44389`), OpenAI Chat `ChatCompletionMessageToolCall` (`specs/openai.with-code-samples.yml:34869–34894`), Anthropic `ResponseToolUseBlock` (`specs/anthropic.yml:13563–13588`), Gemini `FunctionCall` (`specs/gemini.json:273–293`).

`ToolCall` canonicalizes these into a flat `(id, name, arguments)` triple where `arguments` is always a parsed `dict`:

| Field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini | `ToolCall` |
|---|---|---|---|---|---|
| ID | `call_id` | `id` | `id` | `id` | `id` |
| Name | `name` | `function.name` | `name` | `name` | `name` |
| Args | `arguments` (JSON string) | `function.arguments` (JSON string) | `input` (object) | `args` (object) | `arguments` (dict) |

In [ ]:
#| export
@dataclass
class ToolCall:
    "Normalized tool call."
    id: str
    name: str
    arguments: dict = field(default_factory=dict)
    server: bool = False
    extra: dict = field(default_factory=dict)

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

### anthropic_tool_calls

In [ ]:
#| export
ant_tc_types = ("tool_use", "server_tool_use", "mcp_tool_use")

def anthropic_tool_call(b):
    if b.get("type") in ant_tc_types:
        extra = {k:v for k,v in b.items() if k not in ("type","id","name","input")}
        return ToolCall(id=b["id"], name=b["name"], arguments=b.get("input") or {}, server=b.get("type")!="tool_use", extra=extra)

def anthropic_tool_calls(resp, delta=False):
    "Extract Anthropic tool_use blocks as normalized tool calls."
    out = []
    for b in resp.get("content", []):
        if tc:= anthropic_tool_call(b): out.append(tc)
    return out

User defined:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],
    tools=[{"name": "simple_add", "description": "add numbers",
            "input_schema": sch['function']['parameters']}]
)
anthropic_tool_calls(resp)

[ToolCall(id='toolu_01ConK83KqJAFezUYCbQXFmw', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}),
 ToolCall(id='toolu_01GBeFMh8XUgQEVqfRD2CC17', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'caller': {'type': 'direct'}})]

Web search:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "Use web search to get the weather in Istanbul"}],
    tools=[{"type": "web_search_20250305", "name": "web_search"}]
)
anthropic_tool_calls(resp)

[ToolCall(id='srvtoolu_01A6kzMiDX7ZGzsW1nPp5ana', name='web_search', arguments={'query': 'Istanbul weather today'}, server=True, extra={})]

MCP:

In [ ]:
anthropic_tool_calls({"content": [
    {"type": "mcp_tool_use", "id": "mcptoolu_abc123", "name": "get_weather",
     "input": {"city": "Istanbul"}, "server_name": "weather-server"}
]})

[ToolCall(id='mcptoolu_abc123', name='get_weather', arguments={'city': 'Istanbul'}, server=True, extra={'server_name': 'weather-server'})]

### openai_responses_tool_calls

In [ ]:
#| export
def openai_responses_tool_call(item):
    typ = item.get("type", "")
    if typ == "function_call":
        extra = {k:v for k,v in item.items() if k not in ("id","name","arguments")}
        return ToolCall(id=item["id"], name=item["name"], 
                        arguments=json.loads(item["arguments"]), extra=extra)
    if typ.endswith("_call"):  # web_search_call, file_search_call, etc.
        name = typ.removesuffix("_call")
        extra = {k:v for k,v in item.items() if k not in ("id")}
        return ToolCall(id=item.get("id",""), name=name, arguments=item.get("action", {}), server=True, extra=extra)

def openai_responses_tool_calls(resp):
    "Extract Responses API tool call items as normalized tool calls."
    out = []
    for item in resp.get('output', []):
        if tc := openai_responses_tool_call(item): out.append(tc)
    return out

User defined:

In [ ]:
resp = await oai_cli.responses.create_response(
    model='gpt-4o-mini',
    input="What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.",
    tools=[{"type": "function", **sch['function']}]
)
openai_responses_tool_calls(resp)

[ToolCall(id='fc_07079e5b966f6d8b0069eb7881b63081a081d4528e9b501c06', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_WVJMupWoABKhiYqAFUoj2wPX'}),
 ToolCall(id='fc_07079e5b966f6d8b0069eb7881b63c81a0aac92589ffa58a58', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_gwfIaqLA4Vkg24uaWHg0zFsk'})]

Web search:

In [ ]:
resp = await oai_cli.responses.create_response(
    model='gpt-4o-mini',
    input="What is the weather in San Francisco today?",
    tools=[{"type": "web_search_preview"}]
)
openai_responses_tool_calls(resp)

[ToolCall(id='ws_0f62d837734bd3ca0069ef473f0ee881919744088c19bfcd1f', name='web_search', arguments={'type': 'search', 'queries': ['San Francisco weather today'], 'query': 'San Francisco weather today'}, server=True, extra={'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['San Francisco weather today'], 'query': 'San Francisco weather today'}})]

File vector search:

In [ ]:
openai_responses_tool_calls({"output": [
    {"type": "file_search_call", "id": "fs_abc123", "status": "completed",
     "queries": ["quarterly revenue"], "results": [{"file_id": "file-xyz", "text": "Revenue was $1M"}]}
]})

[ToolCall(id='fs_abc123', name='file_search', arguments={}, server=True, extra={'type': 'file_search_call', 'status': 'completed', 'queries': ['quarterly revenue'], 'results': [{'file_id': 'file-xyz', 'text': 'Revenue was $1M'}]})]

Web search:

In [ ]:
openai_responses_tool_calls({"output": [
    {"type": "function_call", "call_id": "call_mcp1", "id": "fc_mcp1", "name": "mcp__weather__get_forecast",
     "arguments": '{"city": "Istanbul"}', "status": "completed"}
]})

[ToolCall(id='fc_mcp1', name='mcp__weather__get_forecast', arguments={'city': 'Istanbul'}, server=False, extra={'type': 'function_call', 'call_id': 'call_mcp1', 'status': 'completed'})]

### openai_chat_tool_calls

In [ ]:
#| export
def openai_chat_tool_calls(resp, delta=False):
    "Extract Chat Completions tool calls as normalized tool calls, optionally from streaming delta events"
    out = []
    key = 'delta' if delta else 'message'
    if not   (tcs:= nested_idx(resp, 'choices', 0, key, 'tool_calls')): return out
    for tc in tcs:
        if not (fn:=tc.get("function")): continue
        extra = {k:v for k,v in tc.items() if k not in ("id","function")}
        args  = json.loads(fn.get("arguments")) if not delta else {'_delta': fn.get("arguments")}
        out.append(ToolCall(id=tc.get("id",""), name=fn.get("name",""), arguments=args, extra=extra))
    return out

User defined:

In [ ]:
resp = await oai_cli.chat.create_chat_completion(
    model='gpt-4o-mini',
    messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],
    tools=[sch]
)
openai_chat_tool_calls(resp)

[ToolCall(id='call_yY8RSMwXW6lvsdUiFq1rqWTr', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function'}),
 ToolCall(id='call_lyxh05ANtgEcIA9gcVGEkavV', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function'})]

### gemini_tool_calls

Extract Gemini functionCall parts as normalized tool calls.

As a general rule, if you receive a thought signature in a model response, you should pass it back exactly as received when sending the conversation history in the next turn. When using Gemini 3 models, you must pass back thought signatures during function calling, otherwise you will get a validation error (4xx status code). This includes when using the minimal thinking level setting for Gemini 3 Flash.

from: https://ai.google.dev/gemini-api/docs/thought-signatures

In [ ]:
#| export
def gemini_tool_calls(resp):
    "Extract Gemini functionCall parts as normalized tool calls."
    out = []
    for i,p in enumerate(nested_idx(resp, 'candidates', 0, 'content', 'parts') or []):
        if not (fc:=p.get("functionCall")): continue
        extra = {k:v for k,v in p.items() if k != "functionCall"}
        out.append(ToolCall(id=fc.get("id", f"call_{i}"), name=fc.get("name", ""), arguments=fc.get("args") or {}, extra=extra))
    return out

User defined:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[{"role": "user", "parts": [{"text": "What is 3 + 5 and 10+ 20? Use the simple_add tool in parallel."}]}],
    tools=[{"functionDeclarations": [sch['function']]}]
)
gemini_tool_calls(resp)

[ToolCall(id='mqa834kq', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'thoughtSignature': 'EtICCs8CAQw51sc6mgqlDQrSy8PGRpqT/CAa8k3y91U17/o5lB2M0r1CT31GhWiL/ZDg6uNptErERw3Tc2MBdzgG2Y9UE/RdE02sz3U6hwIUiduazNMTTNOyscJoxiH53aenOTvxBinUYFFO0Yu8av1FOSPW3FmIbmGyp22wmrkzFE7W31GZUYV2K89jtKJI4XJkP7WrDBknCz4td7sXx9IyjAf3PJVDIWBdnCkM/1OzxBgEAL5yY3o/f0gPHewo8LsBCZ/KRK2F/WQh1Bq1rn2HsSyuBCqVhxzbzc+P/xAgfhG1zSzqbFp75RIlUpvMB8zIR2wNIu2aZJ5NgNmEuPZcNghRGpd3Dvla5+LCZpvbHvNGNccA6SZyNp5KzMCOT+yHMtiQuhHTDTPxRMyqY+a79V3uGGTu2e6AvA8pqXUca4yCYWE8s1dVyCUmlQcZGDGaURs='}),
 ToolCall(id='ba3gimpc', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={})]

In [ ]:
model_parts = [{"functionCall": {"name": tc.name, "args": tc.arguments}, **tc.extra} for tc in  gemini_tool_calls(resp)]
func_results = [
    {"functionResponse": {"name": "simple_add", "response": {"result": 8}}},
    {"functionResponse": {"name": "simple_add", "response": {"result": 30}}},
]
resp_gem = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[
        {"role": "user",  "parts": [{"text": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]},
        {"role": "model", "parts": model_parts},
        {"role": "user",  "parts": func_results},
    ],
    tools=[{"functionDeclarations": [sch['function']]}]
)
resp_gem

{'candidates': [{'content': {'parts': [{'text': 'OK. 3 + 5 is 8, and 10 + 20 is 30.'}],
    'role': 'model'},
   'finishReason': 'STOP',
   'index': 0}],
 'usageMetadata': {'promptTokenCount': 232,
  'candidatesTokenCount': 24,
  'totalTokenCount': 256,
  'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 232}]},
 'modelVersion': 'gemini-3-flash-preview',
 'responseId': 'Rkfvace8G6LDvdIPipDeuQ8'}

Web search:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=[{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}],
    tools=[{"googleSearch": {}}]
)
gemini_tool_calls(resp)

[]

Code execution:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=[{"role": "user", "parts": [{"text": "Calculate the first 10 fibonacci numbers using code"}]}],
    tools=[{"codeExecution": {}}]
)
gemini_tool_calls(resp)

[]

Computer Use:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[{"role": "user", "parts": [{"text": "Go to google.com and search for 'hello world'"}]}],
    tools=[{"computerUse": {"environment": "ENVIRONMENT_BROWSER"}}]
)
gemini_tool_calls(resp)

[ToolCall(id='otbkk94n', name='open_web_browser', arguments={}, server=False, extra={'thoughtSignature': 'Ev8BCvwBAQw51sec6jqeeIi0Tt8UKqnO6gxOcwO/YLz6xpCezoC/shUvP9kBg4Gd4NjsqAOioYL7h7mCSRrdBlAg0X/WjtmFN5XxTlXOwcCz+jseXbKlYq3vMU46xDq/cnlQ/c+SYl9Lhc1kpxN1zlpyWJtmv3BWP30r5g+CcmxfkRIrtTmNepPdlEBegN5I+PU3pQ4AcJRQVFOkZjwj23vcF04rJSdRZ3siFSv3Y+kGSs0PmLGC2niVMuZ3Dum607KtxSvwBYNDhdIQ6j6RXKlsyPZfSLtEK1vVGf2jGdRBF133ORkEAILL3HNjcD1kQbqfqn3j8AZdVYxxbSpfYlJq'})]

## Usage

All four providers report token usage with different field names and structures:

- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — `ResponseUsage`: `{input_tokens, output_tokens, total_tokens}` with `input_tokens_details.cached_tokens` and `output_tokens_details.reasoning_tokens`.
- **[OpenAI-compat (Chat)](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — `CompletionUsage`: `{prompt_tokens, completion_tokens, total_tokens}` with `prompt_tokens_details.cached_tokens` and `completion_tokens_details.reasoning_tokens`.
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — `Usage`: `{input_tokens, output_tokens}` (no total) with `cache_read_input_tokens`, `cache_creation_input_tokens`, and `server_tool_use`.
- **[Gemini](https://ai.google.dev/api/generate-content)** — `UsageMetadata`: `{promptTokenCount, candidatesTokenCount, totalTokenCount}` with `cachedContentTokenCount`, `thoughtsTokenCount`, and `toolUsePromptTokenCount`.

> **Spec sources:** OpenAI Responses `ResponseUsage` (`specs/openai.with-code-samples.yml:62283–62317`), OpenAI Chat `CompletionUsage` (`specs/openai.with-code-samples.yml:36031–36080`), Anthropic `Usage` (`specs/anthropic.yml:14459–14508`), Gemini `UsageMetadata` (`specs/gemini.json:2200–2236`).

`Usage` normalizes these into a canonical `(prompt_tokens, completion_tokens, total_tokens, raw)` shape:

| Canonical field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini |
|---|---|---|---|---|
| `prompt_tokens` | `input_tokens` | `prompt_tokens` | `input_tokens` | `promptTokenCount` |
| `completion_tokens` | `output_tokens` | `completion_tokens` | `output_tokens` | `candidatesTokenCount` |
| `total_tokens` | `total_tokens` | `total_tokens` | (computed: in + out) | `totalTokenCount` |

**`Usage.raw` preserves the full provider payload** — advanced accounting fields (caching, reasoning, tool use tokens) vary too much across providers to normalize, so downstream code like `estimate_cost` extracts them from `raw` with best-effort multi-key lookups (e.g. `_cached_prompt_tokens` tries `cached_input_tokens`, `cache_read_input_tokens`, `prompt_tokens_details.cached_tokens`).

In [ ]:
#| export
@dataclass(frozen=True)
class Usage:
    "Normalized usage."
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    cached_tokens: int = 0
    cache_creation_tokens: int = 0
    reasoning_tokens: int = 0
    raw: dict = field(default_factory=dict)


**TODO:** we'll create a callback system which will compute costs for all 4 providers and letting users to pass a cost mapping for a given model, or we'll let the users to write the callback having access to raw usage.

**Note:** We can probably just store the raw version, and let users write callback functions for a given model to compute usage/cost etc..

### usage_from_anthropic

Normalize Anthropic usage shape.

In [ ]:
#| export
def usage_from_anthropic(resp):
    "Normalize Anthropic usage shape."
    if not (usg:=resp.get("usage")): return None
    cached = int(usg.get("cache_read_input_tokens", 0) or 0)
    cache_creation = int(usg.get("cache_creation_input_tokens", 0) or 0)
    pt = int(usg.get("input_tokens", 0) or 0) + cached + cache_creation
    ct = int(usg.get("output_tokens", 0) or 0)
    return Usage(prompt_tokens=pt, completion_tokens=ct, total_tokens=pt + ct,
                 cached_tokens=cached, cache_creation_tokens=cache_creation, raw=usg)


Text response:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "Hi"}]
)
usage_from_anthropic(resp)

Usage(prompt_tokens=8, completion_tokens=20, total_tokens=28, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 8, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 20, 'service_tier': 'standard', 'inference_geo': 'not_available'})

Tool call response:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],
    tools=[{"name": "simple_add", "description": "add numbers",
            "input_schema": sch['function']['parameters']}]
)
usage_from_anthropic(resp)

Usage(prompt_tokens=417, completion_tokens=138, total_tokens=555, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 417, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 138, 'service_tier': 'standard', 'inference_geo': 'not_available'})

Web search response:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "Use web search to get the weather in Istanbul"}],
    tools=[{"type": "web_search_20250305", "name": "web_search"}]
)
usage_from_anthropic(resp)

Usage(prompt_tokens=11418, completion_tokens=255, total_tokens=11673, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 11418, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 255, 'service_tier': 'standard', 'inference_geo': 'not_available', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}})

### usage_from_openai

Normalize OpenAI usage shape(s)

In [ ]:
#| export
def usage_from_openai(resp):
    "Normalize OpenAI usage shape(s)."
    if not (usg:=resp.get("usage")): return None
    pt = int(usg.get("prompt_tokens", usg.get("input_tokens", 0)) or 0)
    ct = int(usg.get("completion_tokens", usg.get("output_tokens", 0)) or 0)
    tt = int(usg.get("total_tokens", pt + ct) or (pt + ct))
    pd = usg.get("prompt_tokens_details") or usg.get("input_tokens_details") or {}
    cd = usg.get("completion_tokens_details") or usg.get("output_tokens_details") or {}
    cached = int(pd.get("cached_tokens", 0) or 0)
    reasoning = int(cd.get("reasoning_tokens", 0) or 0)
    server_tool_use = dict(Counter(o['type'] for o in resp.get('output', []) if o.get('type') != 'function_call' and o.get('type', '').endswith('_call')))
    if server_tool_use: usg['server_tool_use'] = server_tool_use
    return Usage(prompt_tokens=pt, completion_tokens=ct, total_tokens=tt,
                 cached_tokens=cached, reasoning_tokens=reasoning, raw=usg)


Yes you don't have access to make api calls, just inspect them with `doc` and write me the code!

Text response:

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model='gpt-4o-mini', messages=[{"role": "user", "content": "Say hi"}])
usage_from_openai(resp)

Usage(prompt_tokens=9, completion_tokens=10, total_tokens=19, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 9, 'completion_tokens': 10, 'total_tokens': 19, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})

In [ ]:
resp = await oai_cli.chat.create_chat_completion(
    model='gpt-4o-mini',
    messages=[{"role": "user", "content": "Hi!"}],
    tools=[sch]
)
usage_from_openai(resp)

Usage(prompt_tokens=46, completion_tokens=10, total_tokens=56, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 46, 'completion_tokens': 10, 'total_tokens': 56, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})

Tool call response:

In [ ]:
resp = await oai_cli.responses.create_response(
    model='gpt-4o-mini',
    input="What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.",
    tools=[{"type": "function", **sch['function']}]
)
usage_from_openai(resp)

Usage(prompt_tokens=60, completion_tokens=53, total_tokens=113, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 60, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 53, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 113})

In [ ]:
resp = await oai_cli.chat.create_chat_completion(
    model='gpt-4o-mini',
    messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],
    tools=[sch]
)
usage_from_openai(resp)

Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})

Server tools:

In [ ]:
resp = await oai_cli.responses.create_response(
    model='gpt-4o-mini',
    input="What is the weather in San Francisco today?",
    tools=[{"type": "web_search_preview"}]
)
usage_from_openai(resp)

Usage(prompt_tokens=310, completion_tokens=525, total_tokens=835, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 310, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 525, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 835, 'server_tool_use': {'web_search_call': 1}})

### usage_from_gemini

Normalize Gemini usageMetadata shape.

In [ ]:
#| export
def usage_from_gemini(resp):
    "Normalize Gemini usageMetadata shape."
    if not (usg:=resp.get("usageMetadata")): return None
    pt = int(usg.get("promptTokenCount", 0) or 0)
    ct = int(usg.get("candidatesTokenCount", 0) or 0)
    tt = int(usg.get("totalTokenCount", pt + ct) or (pt + ct))
    cached = int(usg.get("cachedContentTokenCount", 0) or 0)
    reasoning = int(usg.get("thoughtsTokenCount", 0) or 0)
    parts = nested_idx(resp, 'candidates', 0, 'content', 'parts') or []
    cand = nested_idx(resp, 'candidates', 0) or {}
    stu = {}
    if any("executableCode" in p for p in parts): stu["code_execution"] = sum(1 for p in parts if "executableCode" in p)
    if "groundingMetadata" in cand: stu["google_search"] = 1
    if stu: usg['server_tool_use'] = stu
    return Usage(prompt_tokens=pt, completion_tokens=ct, total_tokens=tt,
                 cached_tokens=cached, reasoning_tokens=reasoning, raw=usg)


Text response:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=[{"role": "user", "parts": [{"text": "hi!"}]}],
)
usage_from_gemini(resp)

Usage(prompt_tokens=3, completion_tokens=10, total_tokens=34, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=21, raw={'promptTokenCount': 3, 'candidatesTokenCount': 10, 'totalTokenCount': 34, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 3}], 'thoughtsTokenCount': 21})

Server tools:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=[{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}],
    tools=[{"googleSearch": {}}]
)
usage_from_gemini(resp)

Usage(prompt_tokens=9, completion_tokens=160, total_tokens=403, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=137, raw={'promptTokenCount': 9, 'candidatesTokenCount': 160, 'totalTokenCount': 403, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 9}], 'toolUsePromptTokenCount': 97, 'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 97}], 'thoughtsTokenCount': 137, 'server_tool_use': {'google_search': 1}})

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=[{"role": "user", "parts": [{"text": "Calculate the first 10 fibonacci numbers using code"}]}],
    tools=[{"codeExecution": {}}]
)
usage_from_gemini(resp)

Usage(prompt_tokens=12, completion_tokens=142, total_tokens=405, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=48, raw={'promptTokenCount': 12, 'candidatesTokenCount': 142, 'totalTokenCount': 405, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 12}], 'toolUsePromptTokenCount': 203, 'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 203}], 'thoughtsTokenCount': 48, 'server_tool_use': {'code_execution': 1}})

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=[{"role": "user", "parts": [{"text": "What is solveit? https://solve.it.com/"}]}],
    tools=[{"urlContext": {}}]
)
usage_from_gemini(resp)

Usage(prompt_tokens=14, completion_tokens=311, total_tokens=3451, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=33, raw={'promptTokenCount': 14, 'candidatesTokenCount': 311, 'totalTokenCount': 3451, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 14}], 'toolUsePromptTokenCount': 3093, 'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 3093}], 'thoughtsTokenCount': 33, 'server_tool_use': {'google_search': 1}})

In [ ]:
print(nested_idx(resp, 'candidates', 0, 'content', 'parts', 0, 'text'))



Solveit is a method and a software platform designed to help people become better problem solvers, clearer thinkers, and more elegant coders by leveraging AI without outsourcing their thinking to it. It's inspired by George Pólya's "How to Solve It" and the fast.ai educational tradition. The Solveit method focuses on building in small, iterative steps with immediate feedback. The Solveit platform combines features from tools like ChatGPT, Jupyter, Claude Code, and Cursor, offering a cloud-based Linux development and writing environment with AI integration. It is taught through a 5-week (10-lesson) course that covers topics such as data structures, algorithms, web programming, writing, reading, using web APIs, system administration, and building startups. The course includes access to the Solveit platform and is led by Jeremy Howard from fast.ai and Eric Ries, creator of The Lean Startup.


## Completion

Canonical non-stream model response object

All four providers return non-stream responses in different shapes. `Completion` normalizes these into a canonical `(model, message, finish_reason, usage, tool_calls, raw)` object.

**Provider response → `Completion` field mapping:**

| Field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini |
|---|---|---|---|---|
| `model` | `raw.model` | `raw.model` | `raw.model` | passthrough (input model) |
| `message` | `output[].content[]` → `Msg(role="assistant", content=[Part...])` | `choices[0].message.content` → `Msg` | `content[]` → `Msg` | `candidates[0].content.parts[]` → text joined into single `Part` |
| `finish_reason` | `raw.status` (e.g. `"completed"`) | `choices[0].finish_reason` (e.g. `"stop"`) | `raw.stop_reason` (e.g. `"end_turn"`) | `candidates[0].finishReason` (e.g. `"STOP"`) |
| `usage` | `usage_from_openai(raw.usage)` | `usage_from_openai(raw.usage)` | `usage_from_anthropic(raw.usage)` | `usage_from_gemini(raw.usageMetadata)` |
| `tool_calls` | `output[]` where `type=="function_call"` → flat `ToolCall` | `message.tool_calls[].function` → `ToolCall` | `content[]` where `type=="tool_use"` → `ToolCall` | `parts[]` with `functionCall` → `ToolCall` |
| `raw` | full response dict | full response dict | full response dict | full response dict |

**Notable differences:**
- **Anthropic** puts tool calls in *both* `message.content` (as `Part(type="tool_use")`) *and* `tool_calls` — dual representation for downstream flexibility
- **Gemini** joins all text parts into a single string rather than preserving individual parts
- **Gemini** uses the input `model` string directly since the response only contains `modelVersion` (a version identifier, not a model name)

In [ ]:
#| export
@dataclass(frozen=True)
class Completion:
    "Normalized completion response."
    model: str
    message: Msg
    finish_reason: str = None
    usage: Usage = None
    tool_calls: List[ToolCall] = field(default_factory=list)
    api_name: str = None
    vendor_name: str = None
    raw: dict = field(default_factory=dict)

## Helpers

### Finish Reason

In [ ]:
#| export
ApiName = str_enum('api_name', 'openai', 'openai_chat', 'anthropic', 'gemini')
FinishReason = str_enum('finish_reason', 'stop', 'tool_calls', 'length', 'content_filter')

finish_reps_map = {}
finish_reps_map[ApiName.openai]    = {'completed':'stop', 'incomplete':'length', 'failed':'content_filter'}
finish_reps_map[ApiName.anthropic] = {'end_turn':'stop', 'tool_use':'tool_calls', 'max_tokens':'length'}
finish_reps_map[ApiName.gemini]    = {'STOP':'stop', 'MAX_TOKENS':'length', 'SAFETY':'content_filter', 'BLOCKLIST':'content_filter'}

def canon_finish(reason, api_name, tcs=None):
    "Canonicalize finish_reason to OpenAI Chat values: stop, tool_calls, length, content_filter."
    r =  finish_reps_map.get(api_name, {}).get(reason, reason)
    return FinishReason.tool_calls if r==FinishReason.stop and any(~L(tcs).attrgot('server')) else r 

In [ ]:
test_eq(canon_finish('completed','openai') == canon_finish('end_turn','anthropic') == canon_finish('STOP','gemini') == 'stop', True)

In [ ]:
test_eq(canon_finish('completed','openai', ToolCall('123', 'f', {})), 'tool_calls')
test_eq(canon_finish('completed','openai', ToolCall('123', 'web_search', {}, server=True)), 'stop')

## OpenAI Responses

### normalize_openai_response

Normalize OpenAI Responses API object into Completion.

**NB:** Per the OpenAI spec, `text`, `refusal`, and `summary_text` fields are all **required** — the `.get(k, "")` fallbacks are purely defensive against malformed responses. `OutputMessageContent` is a discriminated `oneOf` with exactly two variants: `OutputTextContent` (`output_text`) and `RefusalContent` (`refusal`) — no other content types exist in message blocks.

`summary` is an array of `SummaryTextContent` only — no discriminated union, just one type. So the `if s.get("type") == "summary_text"` check is technically redundant (it'll always be `summary_text`), and `text` is required.

Additionally, `Response.output`, `OutputMessage.content`, and `ReasoningItem.summary` are all **required** fields — so `.get()` with fallback defaults is defensive but not strictly necessary.


In [ ]:
#| export
def openai_responses_parts(resp):
    parts = []
    for item in resp["output"]:
        if (typ:=item["type"]) == "message":
            for c in item["content"]:
                if (ctyp := c["type"]) == "output_text":
                    c['citations'] = c.pop('annotations', [])
                    parts.append(Part(type=PartType.text, text=c['text'], data=c))
                elif ctyp == "refusal":
                    parts.append(Part(type=PartType.refusal, text=c['refusal'], data=c))
        elif typ == "reasoning":
            for s in item["summary"]: parts.append(Part(type=PartType.thinking, text=s['text'], data=item))
        elif typ == "function_call" or typ.endswith("_call"):
            tc = openai_responses_tool_call(item)
            tdata = {**tc.extra, 'id':tc.id, 'name':tc.name, 'arguments':tc.arguments, 'server':tc.server}
            parts.append(Part(type=PartType.tool_use, data=tdata))
    return parts

In [ ]:
#| export
def normalize_openai_response(resp, model, api_name=ApiName.openai, vendor_name="openai"):
    "Normalize OpenAI Responses API object into Completion."
    tcs = openai_responses_tool_calls(resp)
    return Completion(
        model=resp.get("model") or model,
        message=Msg(role="assistant", content=openai_responses_parts(resp)),
        finish_reason=canon_finish(resp.get("status"), api_name, tcs),
        usage=usage_from_openai(resp),
        tool_calls=tcs,
        api_name=api_name,
        vendor_name=vendor_name,
        raw=resp)

#### Text

In [ ]:
mn = 'gpt-4o-mini'
c = normalize_openai_response(await oai_cli.responses.create_response(model=mn, input="Say hi"), mn)
c.message.content, c.finish_reason


([Part(type=<PartType.text: 'text'>, text='Hi there! How can I help you today?', data={'type': 'output_text', 'logprobs': [], 'text': 'Hi there! How can I help you today?', 'citations': []})],
 'stop')

#### Tool call

In [ ]:
resp = await oai_cli.responses.create_response(model=mn,
    input="What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.",
    tools=[{"type": "function", **sch['function']}]
)
c = normalize_openai_response(resp, model=mn)
test_eq(c.finish_reason, 'tool_calls')
test_eq(len(c.message.content), 2)
c.tool_calls, c.message.content

([ToolCall(id='fc_07079e5b966f6d8b0069eb7881b63081a081d4528e9b501c06', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_WVJMupWoABKhiYqAFUoj2wPX'}),
  ToolCall(id='fc_07079e5b966f6d8b0069eb7881b63c81a0aac92589ffa58a58', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_gwfIaqLA4Vkg24uaWHg0zFsk'})],
 [Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'type': 'function_call', 'status': 'completed', 'call_id': 'call_WVJMupWoABKhiYqAFUoj2wPX', 'id': 'fc_07079e5b966f6d8b0069eb7881b63081a081d4528e9b501c06', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False}),
  Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'type': 'function_call', 'status': 'completed', 'call_id': 'call_gwfIaqLA4Vkg24uaWHg0zFsk', 'id': 'fc_07079e5b966f6d8b0069eb7881b63c81a0aac92589ffa58a58', 'name': 'simple_add', 'ar

Web search response:

In [ ]:
resp = await oai_cli.responses.create_response(model=mn,
    input="What is the weather in Istanbul today?",
    tools=[{"type": "web_search_preview"}]
)
c = normalize_openai_response(resp, model=mn)
c.tool_calls, next(p.text for p in c.message.content if p.type == 'text')[:100]

([ToolCall(id='ws_014a600da66b942c0069eb7884e6fc8191ad64ffed70b7f64b', name='web_search', arguments={'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}, server=True, extra={'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}})],
 'As of 5:04 PM local time in Istanbul on April 24, 2026, the weather is sunny with a temperature of 6')

In [ ]:
test_eq(c.tool_calls[0].server, True)
test_eq(c.finish_reason, 'stop')

Refusal response (mocked — hard to trigger reliably via API):

In [ ]:
c = normalize_openai_response({"model": "gpt-4o-mini", "status": "completed", "output": [
    {"type": "message", "role": "assistant", "content": [
        {"type": "refusal", "refusal": "I can't help with that request."}
    ]}
], "usage": {"input_tokens": 10, "output_tokens": 5, "total_tokens": 15}}, model='gpt-4o-mini')
c.message.content


[Part(type=<PartType.refusal: 'refusal'>, text="I can't help with that request.", data={'type': 'refusal', 'refusal': "I can't help with that request."})]

## OpenAI Chat

### normalize_openai_chat_completion

Normalize chat.completions response object into Completion.

In [ ]:
#| export
def normalize_openai_chat_completion(resp, model, api_name=ApiName.openai_chat, vendor_name="openai"):
    "Normalize chat.completions response object into Completion."
    msg = nested_idx(resp, 'choices', 0, 'message') or {}
    parts = []
    if thinking := msg.get('reasoning_content'): parts.append(Part(type="thinking", text=thinking))
    if cts := msg.get('content'): parts.append(Part(type="text",text=cts,data=dict(citations=msg.get('annotations',[]))))
    if ref := msg.get('refusal'): parts.append(Part(type="refusal",text=ref))
    tcs = openai_chat_tool_calls(resp)
    for tc in tcs: 
        tdata = {**tc.extra, 'id':tc.id, 'name':tc.name, 'arguments':tc.arguments, 'server':tc.server}
        parts.append(Part(type="tool_use", data=tdata))
    return Completion(
        model=resp.get("model") or model,
        message=Msg(role="assistant", content=parts),
        finish_reason=nested_idx(resp, 'choices', 0, 'finish_reason'),
        usage=usage_from_openai(resp),
        tool_calls=tcs,
        api_name=api_name,
        vendor_name=vendor_name,
        raw=resp)

#### Text

In [ ]:
resp = await oai_cli.chat.create_chat_completion(
    model='gpt-4o-mini', messages=[{"role": "user", "content": "Say hi"}]
)
c = normalize_openai_chat_completion(resp, model='gpt-4o-mini')
c.message.content, c.finish_reason

([Part(type='text', text='Hi there! How can I assist you today?', data={'citations': []})],
 'stop')

#### Tool call

In [ ]:
resp = await oai_cli.chat.create_chat_completion(
    model='gpt-4o-mini',
    messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],
    tools=[sch]
)
c = normalize_openai_chat_completion(resp, model='gpt-4o-mini')
c.tool_calls, c.message.content, c.finish_reason

([ToolCall(id='call_yY8RSMwXW6lvsdUiFq1rqWTr', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function'}),
  ToolCall(id='call_lyxh05ANtgEcIA9gcVGEkavV', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function'})],
 [Part(type='tool_use', text=None, data={'type': 'function', 'id': 'call_yY8RSMwXW6lvsdUiFq1rqWTr', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False}),
  Part(type='tool_use', text=None, data={'type': 'function', 'id': 'call_lyxh05ANtgEcIA9gcVGEkavV', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False})],
 'tool_calls')

#### Refusal (mocked)

In [ ]:
c = normalize_openai_chat_completion({"model": "gpt-4o-mini", "choices": [
    {"index": 0, "message": {"role": "assistant", "content": None, "refusal": "I can't help with that."},
     "finish_reason": "stop"}
], "usage": {"prompt_tokens": 10, "completion_tokens": 5, "total_tokens": 15}}, model='gpt-4o-mini')
c.message.content, c.finish_reason

([Part(type='refusal', text="I can't help with that.", data=None)], 'stop')

## Anthropic messages

### normalize_anthropic_message

Normalize Anthropic message response into Completion

In [ ]:
#| export
def _ant_part_type(typ):
    "Map Anthropic content block type to canonical PartType."
    ant_parts_mapping = dict(text=PartType.text, thinking=PartType.thinking, redacted_thinking=PartType.thinking, tool_use=PartType.tool_use, server_tool_use=PartType.tool_use, mcp_tool_use=PartType.tool_use)
    if typ in ant_parts_mapping: return ant_parts_mapping[typ]
    if typ.endswith('_tool_result'): return PartType.server_tool_result
    return typ

def normalize_anthropic_message(resp, model, api_name=ApiName.anthropic, vendor_name='anthropic'):
    "Normalize Anthropic message response into Completion."
    tcs = anthropic_tool_calls(resp)
    tc_map = {tc.id: tc for tc in tcs}
    parts = []
    for b in resp.get("content", []):
        typ = _ant_part_type(b.get("type", "text"))
        if typ == PartType.thinking: parts.append(Part(type=PartType.thinking, text=b.get("thinking", ""), data=b))
        elif typ == "tool_use":
            if tc:=tc_map.get(b.get("id")):
                tdata = {**tc.extra, 'id':tc.id, 'name':tc.name, 'arguments':tc.arguments, 'server':tc.server}
                parts.append(Part(type=PartType.tool_use, data=tdata))
        else: parts.append(Part(type=typ, text=b.get("text", ""), data=b))
    return Completion(
        model=resp.get("model") or model,
        message=Msg(role="assistant", content=parts),
        finish_reason=canon_finish(resp.get("stop_reason"), 'anthropic'),
        usage=usage_from_anthropic(resp),
        tool_calls=tcs,
        api_name=api_name,
        vendor_name=vendor_name,
        raw=resp)

#### Text

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "Say hi"}]
)
c = normalize_anthropic_message(resp, model='claude-sonnet-4-20250514')
c.message.content, c.finish_reason

([Part(type=<PartType.text: 'text'>, text='Hi there! How are you doing today?', data={'type': 'text', 'text': 'Hi there! How are you doing today?'})],
 'stop')

#### Tool call

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],
    tools=[{"name": "simple_add", "description": "add numbers",
            "input_schema": sch['function']['parameters']}]
)
c = normalize_anthropic_message(resp, model='claude-sonnet-4-20250514')
c.tool_calls, c.message.content, c.finish_reason

([ToolCall(id='toolu_01ConK83KqJAFezUYCbQXFmw', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}),
  ToolCall(id='toolu_01GBeFMh8XUgQEVqfRD2CC17', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'caller': {'type': 'direct'}})],
 [Part(type=<PartType.text: 'text'>, text="I'll calculate both additions using the simple_add tool in parallel.", data={'type': 'text', 'text': "I'll calculate both additions using the simple_add tool in parallel."}),
  Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'caller': {'type': 'direct'}, 'id': 'toolu_01ConK83KqJAFezUYCbQXFmw', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False}),
  Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'caller': {'type': 'direct'}, 'id': 'toolu_01GBeFMh8XUgQEVqfRD2CC17', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False})],
 'tool_calls')

#### Thinking (mocked)

In [ ]:
c = normalize_anthropic_message({"model": "claude-sonnet-4-20250514", "content": [
    {"type": "thinking", "thinking": "Let me reason...", "text": ""},
    {"type": "redacted_thinking", "text": ""},
    {"type": "text", "text": "Here's my answer."}
], "stop_reason": "end_turn", "usage": {"input_tokens": 10, "output_tokens": 20}}, model='claude-sonnet-4-20250514')
[(p.type, p.text[:30]) for p in c.message.content]

[(<PartType.thinking: 'thinking'>, 'Let me reason...'),
 (<PartType.thinking: 'thinking'>, ''),
 (<PartType.text: 'text'>, "Here's my answer.")]

#### Server tool result (mocked)

In [ ]:
c = normalize_anthropic_message({"model": "claude-sonnet-4-20250514", "content": [
    {"type": "server_tool_use", "id": "srvtoolu_abc", "name": "web_search", "input": {"query": "weather"}},
    {"type": "web_search_tool_result", "tool_use_id": "srvtoolu_abc", "content": [{"type": "web_search_result", "title": "Weather", "url": "https://example.com"}]},
    {"type": "text", "text": "The weather is sunny."}
], "stop_reason": "end_turn", "usage": {"input_tokens": 100, "output_tokens": 50}}, model='claude-sonnet-4-20250514')
[(p.type, (p.text or '')[:30]) for p in c.message.content], c.tool_calls

([(<PartType.tool_use: 'tool_use'>, ''),
  (<PartType.server_tool_result: 'server_tool_result'>, ''),
  (<PartType.text: 'text'>, 'The weather is sunny.')],
 [ToolCall(id='srvtoolu_abc', name='web_search', arguments={'query': 'weather'}, server=True, extra={})])

#### MCP tool call (mocked)

In [ ]:
c = normalize_anthropic_message({"model": "claude-sonnet-4-20250514", "content": [
    {"type": "mcp_tool_use", "id": "mcptoolu_abc", "name": "get_weather", "input": {"city": "Istanbul"}, "server_name": "weather-server"}
], "stop_reason": "tool_use", "usage": {"input_tokens": 50, "output_tokens": 30}}, model='claude-sonnet-4-20250514')
[(p.type, p.text) for p in c.message.content], c.tool_calls

([(<PartType.tool_use: 'tool_use'>, None)],
 [ToolCall(id='mcptoolu_abc', name='get_weather', arguments={'city': 'Istanbul'}, server=True, extra={'server_name': 'weather-server'})])

#### Web search

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "Use web search to get the weather in Istanbul"}],
    tools=[{"type": "web_search_20250305", "name": "web_search"}]
)
c = normalize_anthropic_message(resp, model='claude-sonnet-4-20250514')
c.tool_calls, [p.type for p in c.message.content]

([ToolCall(id='srvtoolu_01A6kzMiDX7ZGzsW1nPp5ana', name='web_search', arguments={'query': 'Istanbul weather today'}, server=True, extra={})],
 [<PartType.tool_use: 'tool_use'>,
  <PartType.server_tool_result: 'server_tool_result'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>])

#### Empty content (mocked)

In [ ]:
c = normalize_anthropic_message({"model": "claude-sonnet-4-20250514", "content": [],
    "stop_reason": "end_turn", "usage": {"input_tokens": 5, "output_tokens": 0}},
    model='claude-sonnet-4-20250514')
c.message.content, c.finish_reason

([], 'stop')

## Gemini generate

### normalize_gemini_generate

Normalize Gemini generateContent response

In [ ]:
#| export
def _gem_part_type(p):
    "Map Gemini part to canonical PartType."
    if 'functionCall' in p or 'toolCall' in p: return PartType.tool_use
    if 'executableCode' in p or 'codeExecutionResult' in p: return PartType.tool_result
    if p.get('thought'): return PartType.thinking
    return PartType.text

def normalize_gemini_generate(resp, model, api_name=ApiName.gemini, vendor_name='gemini'):
    "Normalize Gemini generateContent response."
    c0 = nested_idx(resp, 'candidates', 0) or {}
    tcs = gemini_tool_calls(resp)
    tc_map = {tc.id: tc for tc in tcs}
    parts = []
    for p in nested_idx(c0, 'content', 'parts') or []:
        typ = _gem_part_type(p)
        if typ == 'tool_use':
            fc = p.get('functionCall') or p.get('toolCall') or {}
            tc = tc_map.get(fc.get('id'))
            if tc: 
                tdata = {**tc.extra, 'id':tc.id, 'name':tc.name, 'arguments':tc.arguments, 'server':tc.server}
                parts.append(Part(type=PartType.tool_use, data=tdata))
        else: parts.append(Part(type=typ, text=p.get("text",""), data=p))
    if citations := c0.get('groundingMetadata'):
        for p in parts: 
            if p.type == PartType.text: p.data['citations'] = citations
    return Completion(
        model=model,
        message=Msg(role="assistant", content=parts),
        finish_reason=canon_finish(c0.get("finishReason"), 'gemini', tcs),
        usage=usage_from_gemini(resp),
        tool_calls=tcs,
        api_name=api_name,
        vendor_name=vendor_name,
        raw=resp)

#### Text

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[{"role": "user", "parts": [{"text": "Say hi in one word"}]}]
)
c = normalize_gemini_generate(resp, model='gemini-2.5-flash')
c.message.content, c.finish_reason

([Part(type=<PartType.text: 'text'>, text='Hi', data={'text': 'Hi', 'thoughtSignature': 'Ev0BCvoBAQw51seShXRLAs5KV5ziMXxwRWEWHPDjgfeqYMiGIw1nkCrWRQIGGtHJH9LoifDNdtno+5WJWfQW9bcT6yDWgC9ngCw8RpScSyWkE8S7HrfM3tXdCYbQOidCPwT9yQ+BbbS3AyC2EbdPjcTF8DwdcXtVsCIN9+7Tdkl3JNbJfvh2svOOD5HaxKETxaMVRiUTTcvxt7/w2HaVq4X5CAiXMEn/j6kMH5KizLNPHmWv++35RC16Lmlq18aQPsP6F20k1wEu1oWrmhWv5im8UevAuLSbSgLWE0gZ3z0lxCjFePwUnl3Mdo41zdv9bISEOkyK6yidmnGZTFux/w=='})],
 'stop')

#### Tool call

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[{"role": "user", "parts": [{"text": "What is 3 + 5? Use simple_add tool."}]}],
    tools=[{"functionDeclarations": [sch['function']]}]
)
c = normalize_gemini_generate(resp, model='gemini-2.5-flash')
c.tool_calls, [(p.type, p.text[:30] if p.text else '') for p in c.message.content], c.finish_reason

([ToolCall(id='0l2t6oy8', name='simple_add', arguments={'b': 5, 'a': 3}, server=False, extra={'thoughtSignature': 'EtoBCtcBAQw51sezE4VfHzXrfo2leBv+yijbB8ofzZcVa41R42IzWPhM9FeXVCl0hiYR2sT0WcCF9zlJ0LziHFIsVzSJXSBDxvRT+1Fla2+BAJ/12hFyYQX9Uc5bYMjnQ91xQSmUuu+ieCsUNuen6mMjNKiVv++1hIs9mHnQ7E1dRNBPSpPu6QgyKxdEUdsjNrAHO+6zQT5JX6KEa4WvEEeLvQIOr5o8xp1QRhVN77dBLKJ/Yv+K5fvSHijccR9D1E8Fxigam0Usa9s8ZJ1qpRJGx4TMuaBfOFegQNg='})],
 [(<PartType.tool_use: 'tool_use'>, '')],
 <finish_reason.tool_calls: 'tool_calls'>)

In [ ]:
c.message.content[0].data

{'thoughtSignature': 'EtoBCtcBAQw51sezE4VfHzXrfo2leBv+yijbB8ofzZcVa41R42IzWPhM9FeXVCl0hiYR2sT0WcCF9zlJ0LziHFIsVzSJXSBDxvRT+1Fla2+BAJ/12hFyYQX9Uc5bYMjnQ91xQSmUuu+ieCsUNuen6mMjNKiVv++1hIs9mHnQ7E1dRNBPSpPu6QgyKxdEUdsjNrAHO+6zQT5JX6KEa4WvEEeLvQIOr5o8xp1QRhVN77dBLKJ/Yv+K5fvSHijccR9D1E8Fxigam0Usa9s8ZJ1qpRJGx4TMuaBfOFegQNg=',
 'id': '0l2t6oy8',
 'name': 'simple_add',
 'arguments': {'b': 5, 'a': 3},
 'server': False}

#### Web search

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}],
    tools=[{"googleSearch": {}}]
)
c = normalize_gemini_generate(resp, model='gemini-2.5-flash')
[(p.type, p.text[:40] if p.text else '') for p in c.message.content], c.finish_reason

([(<PartType.text: 'text'>, 'Today, Friday, April 24, 2026, the weath')],
 'stop')

#### Code execution

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[{"role": "user", "parts": [{"text": "Calculate the first 10 fibonacci numbers using code"}]}],
    tools=[{"codeExecution": {}}]
)
c = normalize_gemini_generate(resp, model='gemini-2.5-flash')
[(p.type, p.text[:40] if p.text else '') for p in c.message.content], c.finish_reason

([(<PartType.text: 'text'>, 'Here is the Python code to calculate the')],
 'stop')

#### Thinking (mocked)

In [ ]:
c = normalize_gemini_generate({"candidates": [{"content": {"parts": [
    {"text": "Let me think...", "thought": True},
    {"text": "Here's the answer."}
], "role": "model"}, "finishReason": "STOP"}],
"usageMetadata": {"promptTokenCount": 10, "candidatesTokenCount": 20, "totalTokenCount": 30}},
model='gemini-2.5-flash')
[(p.type, p.text) for p in c.message.content]

[(<PartType.thinking: 'thinking'>, 'Let me think...'),
 (<PartType.text: 'text'>, "Here's the answer.")]

#### Empty candidates (mocked)

In [ ]:
c = normalize_gemini_generate({"candidates": [],
"usageMetadata": {"promptTokenCount": 5, "candidatesTokenCount": 0, "totalTokenCount": 5}},
model='gemini-2.5-flash')
c.message.content, c.finish_reason

([], None)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()